In [ ]:
# Installation de bibliothèque GEKKO
!pip install gekko

#**Génération du Dataset d'entraînement via la commande prédictive experte (MPC)**

In [ ]:
import numpy as np
import pandas as pd
from gekko import GEKKO

print("Préparation du simulateur MPC...")

# ==========================================
# 1. Initialisation du Contrôleur GEKKO
# ==========================================
m = GEKKO(remote=False)
m.time = np.linspace(0, 150, 31)
# Variable manipulée : puissance de chauffage en %, limitée entre 0 % et 100 %
u = m.MV(value=0, lb=0, ub=100)
u.STATUS = 1
u.DCOST = 0.0001

y = m.CV(value=21.1)
y.STATUS = 1
y.FSTATUS = 1
y.TR_INIT = 1
y.TAU = 5.0   # Lissage de la trajectoire de consigne
y.WSP = 2000.0
# Modèle thermique du premier ordre identifié pour la maquette
m.Equation(441.0 * y.dt() + (y - 21.1) == 0.355 * u)
m.options.IMODE = 6
m.options.CV_TYPE = 2

# ==========================================
# 2. Boucle de Génération
# ==========================================
n_samples = 10000
dataset = []

temp_actuelle = 21.1
temp_t1 = 21.1
temp_t2 = 21.1
consigne = 28.0

print("Création du dataset en cours...")

for i in range(n_samples):
    # Changement de consigne tous les 1000 pas
    if i % 1000 == 0:
        consigne = np.random.uniform(23.0, 35.0)

    # Génération d'une valeur d'humidité simulée comme entrée du modèle
    humidite_actuelle = np.random.uniform(40.0, 95.0)
    # Mise à jour de la consigne et de la mesure actuelle pour le contrôleur MPC
    y.SP = consigne
    y.SPHI = consigne
    y.SPLO = consigne
    y.MEAS = temp_actuelle

    try:
        m.solve(disp=False)
        commande_optimale = u.NEWVAL
    except:
        commande_optimale = u.NEWVAL

    # =======================================================
    # Correction en régime établi pour réduire l'erreur statique
    # =======================================================
    erreur = consigne - temp_actuelle
    if abs(erreur) < 0.15:

        commande_optimale = (consigne - 21.1) / 0.355

    # Saturation de sécurité : garantir une commande entre 0 % et 100 %
    commande_optimale = max(0.0, min(100.0, commande_optimale))

    # Ajout au dataset
    dataset.append([temp_actuelle, consigne, humidite_actuelle, temp_t1, temp_t2, commande_optimale])

    # Mise à jour des températures passées
    temp_t2 = temp_t1
    temp_t1 = temp_actuelle

    # Mise à jour de la température selon le modèle thermique discrétisé
    temp_actuelle = temp_actuelle + (1.0 / 441.0) * (0.355 * commande_optimale - (temp_actuelle - 21.1))

# ==========================================
# 3. Sauvegarde dans le CSV
# ==========================================
colonnes = ['Temp_current', 'Setpoint', 'Humidity', 'Temp_t1', 'Temp_t2', 'Command_MPC']
df = pd.DataFrame(dataset, columns=colonnes)
df.to_csv('data_expert_parfaite.csv', index=False)

print("Succès! Fichier 'data_expert_parfaite.csv' généré.")

#**Architecture et Entraînement du Modèle d'Intelligence Artificielle**

In [ ]:
import pandas as pd
import numpy as np
import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, Dropout
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.callbacks import EarlyStopping
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
import pickle

print("Étape 3 : Entraînement du Réseau de Neurones...")

# ==========================================
# 1. Préparation des données
# ==========================================
# Lecture du dataset expert
df = pd.read_csv('data_expert_parfaite.csv')

# Séparation des Entrées (X) et de la Sortie (y)
X = df[['Temp_current', 'Setpoint', 'Humidity', 'Temp_t1', 'Temp_t2']].values
y = df['Command_MPC'].values

# Normalisation des entrées pour améliorer l'apprentissage du réseau de neurones
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

# Sauvegarde du Scaler pour l'utiliser plus tard sur la Raspberry Pi
with open('scaler.pkl', 'wb') as f:
    pickle.dump(scaler, f)

# Normalisation de la Sortie (y) entre 0 et 1 (car on utilise une activation Sigmoid)
y_scaled = y / 100.0

# Division des données : 70 % entraînement, 15 % validation et 15 % test
X_temp, X_test, y_temp, y_test = train_test_split(X_scaled, y_scaled, test_size=0.15, random_state=42)
X_train, X_val, y_train, y_val = train_test_split(X_temp, y_temp, test_size=(0.15/0.85), random_state=42)

print(f"Données réparties : Train={len(X_train)} | Val={len(X_val)} | Test={len(X_test)}")

# ==========================================
# 2. Construction du Modèle 
# ==========================================
model = Sequential()

# Couche 1: 32 neurones, ReLU, Dropout(0.2) + Input layer (5 entrées)
model.add(Dense(32, input_dim=5, activation='relu'))

# Dropout utilisé pour limiter le surapprentissage sur le dataset simulé
model.add(Dropout(0.2))

# Couche 2: 16 neurones, ReLU, Dropout(0.2)
model.add(Dense(16, activation='relu'))
model.add(Dropout(0.2))

# Couche 3: 8 neurones, ReLU
model.add(Dense(8, activation='relu'))

# Couche de Sortie: 1 neurone, Sigmoid (pour sortir une valeur entre 0 et 1)
model.add(Dense(1, activation='sigmoid'))

# Compilation du modèle
optimizer = Adam(learning_rate=0.001)
model.compile(optimizer=optimizer, loss='mse', metrics=['mae'])

# ==========================================
# 3. Entraînement du Modèle
# ==========================================
# Early stopping pour arrêter l'entraînement si le modèle n'évolue plus
early_stop = EarlyStopping(monitor='val_loss', patience=15, restore_best_weights=True)

print("Début de l'entraînement...")
history = model.fit(
    X_train, y_train,
    validation_data=(X_val, y_val),
    epochs=200,
    batch_size=32,
    callbacks=[early_stop],
    verbose=1
)

# ==========================================
# 4. Conversion en TensorFlow Lite (Pour la Raspberry Pi)
# ==========================================
print("Conversion du modèle en TensorFlow Lite...")

converter = tf.lite.TFLiteConverter.from_keras_model(model)
# Conversion en TFLite sans quantification afin de conserver la précision du modèle
tflite_model = converter.convert()

# Sauvegarder le modèle final
with open('modele_thermique.tflite', 'wb') as f:
    f.write(tflite_model)

print("-" * 40)
print("Succès ! Les fichiers 'modele_thermique.tflite' et 'scaler.pkl' sont prêts.")
print("Les fichiers générés peuvent être déployés sur la Raspberry Pi.")

#**Code d’implémentation sur carte Raspberry Pi**

In [ ]:
import time
import pickle
import numpy as np
import RPi.GPIO as GPIO
import board
import adafruit_dht
import json
import paho.mqtt.client as mqtt

try:
    from tflite_runtime.interpreter import Interpreter
except ImportError:
    from tensorflow.lite.python.interpreter import Interpreter


print("Initialisation du systeme de controle IA...")


# ==========================================
# 0. Configuration MQTT - Compatible Superviseur
# ==========================================
MQTT_BROKER = "localhost"
MQTT_PORT = 1883

MQTT_TOPIC_DATA = "thermal_box/data"

MQTT_TOPIC_SETPOINT = "thermal_box/control/setpoint"
MQTT_TOPIC_MODE = "thermal_box/control/mode"
MQTT_TOPIC_MANUAL = "thermal_box/control/manual_power"
MQTT_TOPIC_APPLY = "thermal_box/control/apply"

PERIODE_MQTT = 2.0

client_mqtt = mqtt.Client(mqtt.CallbackAPIVersion.VERSION2, "RaspberryPi_NNMPC")

try:
    client_mqtt.connect(MQTT_BROKER, MQTT_PORT)
    client_mqtt.loop_start()
    print("-> Connecte au Broker MQTT local avec succes.")
except Exception as e:
    print(f"-> Attention: Impossible de se connecter a MQTT ({e})")

dernier_envoi_mqtt = 0


# ==========================================
# 1. Configuration du Hardware
# ==========================================
PIN_CHAUFFAGE = 25

dhtDevice = adafruit_dht.DHT22(board.D4)

GPIO.setmode(GPIO.BCM)
GPIO.setup(PIN_CHAUFFAGE, GPIO.OUT)
GPIO.output(PIN_CHAUFFAGE, GPIO.LOW)


# ==========================================
# 2. Chargement de l'IA
# ==========================================
with open("scaler.pkl", "rb") as f:
    scaler = pickle.load(f)

interpreter = Interpreter(model_path="modele_thermique.tflite")
interpreter.allocate_tensors()

input_details = interpreter.get_input_details()
output_details = interpreter.get_output_details()


# ==========================================
# 3. Parametres
# ==========================================
CONSIGNE = 24.0
MODE = "auto"
COMMANDE_MANUELLE = 0.0


# ==========================================
# 4. MQTT - Reception commandes dashboard
# ==========================================
def on_message(client, userdata, msg):
    global CONSIGNE, MODE, COMMANDE_MANUELLE

    topic = msg.topic
    payload = msg.payload.decode()

    try:
        if topic == MQTT_TOPIC_SETPOINT:
            CONSIGNE = float(payload)
            print(f"\n>>> [MQTT] Consigne -> {CONSIGNE} C <<<")

        elif topic == MQTT_TOPIC_MODE:
            mode_recu = payload.lower()

            if mode_recu in ["auto", "manual", "schedule"]:
                MODE = mode_recu
                print(f"\n>>> [MQTT] Mode -> {MODE.upper()} <<<")
            else:
                print(f"Erreur : Mode '{payload}' inconnu.")

        elif topic == MQTT_TOPIC_MANUAL:
            COMMANDE_MANUELLE = float(payload) / 100.0
            COMMANDE_MANUELLE = max(0.0, min(1.0, COMMANDE_MANUELLE))
            print(f"\n>>> [MQTT] Puissance manuelle -> {COMMANDE_MANUELLE * 100:.1f}% <<<")

        elif topic == MQTT_TOPIC_APPLY:
            data = json.loads(payload)

            if "setpoint" in data:
                CONSIGNE = float(data["setpoint"])

            if "mode" in data:
                mode_recu = str(data["mode"]).lower()

                if mode_recu in ["auto", "manual", "schedule"]:
                    MODE = mode_recu

            if "manual_power" in data:
                COMMANDE_MANUELLE = float(data["manual_power"]) / 100.0
                COMMANDE_MANUELLE = max(0.0, min(1.0, COMMANDE_MANUELLE))

            print(
                f"\n>>> [MQTT APPLY] Mode={MODE.upper()} | "
                f"Consigne={CONSIGNE} C | "
                f"Manuel={COMMANDE_MANUELLE * 100:.1f}% <<<"
            )

    except Exception as e:
        print(f"Erreur MQTT commande : {e}")


client_mqtt.on_message = on_message

client_mqtt.subscribe([
    (MQTT_TOPIC_SETPOINT, 0),
    (MQTT_TOPIC_MODE, 0),
    (MQTT_TOPIC_MANUAL, 0),
    (MQTT_TOPIC_APPLY, 0)
])


# ==========================================
# 5. Lecture capteur
# ==========================================
def lire_capteur():
    for _ in range(3):
        try:
            temp = dhtDevice.temperature
            hum = dhtDevice.humidity

            if temp is not None and hum is not None:
                return hum, temp

        except RuntimeError:
            time.sleep(2.0)
            continue

        except Exception as error:
            dhtDevice.exit()
            raise error

    return None, None


humidite, temp_actuelle = lire_capteur()

if temp_actuelle is None:
    temp_actuelle = 20.0

if humidite is None:
    humidite = 50.0

temp_t1 = temp_actuelle
temp_t2 = temp_actuelle

print(f"Demarrage de la regulation. Consigne : {CONSIGNE} C")
print("-" * 50)


# ==========================================
# 6. Boucle principale temps reel
# ==========================================
try:
    while True:

        hum, temp = lire_capteur()

        if temp is not None and hum is not None:
            temp_actuelle = temp
            humidite = hum
            sensor_status = "OK"
        else:
            print("Erreur capteur, on garde les anciennes valeurs.")
            sensor_status = "ERROR"

        # ==========================================
        # Controle AUTO / SCHEDULE avec NN-MPC
        # ==========================================
        if MODE == "auto" or MODE == "schedule":

            entrees = np.array([
                [temp_actuelle, CONSIGNE, humidite, temp_t1, temp_t2]
            ])
            # Normalisation des entrées avec le même scaler utilisé pendant l'entraînement
            entrees_scaled = scaler.transform(entrees).astype(np.float32)

            interpreter.set_tensor(input_details[0]["index"], entrees_scaled)
            # Exécution de l'inférence TensorFlow Lite sur Raspberry Pi
            interpreter.invoke()

            commande_predite = interpreter.get_tensor(output_details[0]["index"])[0][0]
            # Sécurité : limitation de la commande entre 0 % et 100 %
            commande_predite = max(0.0, min(1.0, float(commande_predite)))

            # Protection : arrêt du chauffage si la température dépasse la consigne
            if temp_actuelle > CONSIGNE:
                commande_predite = 0.0

            mode_affichage = MODE.upper()
            controller_name = "NN-MPC"

        else:
            commande_predite = COMMANDE_MANUELLE
            mode_affichage = "MANUAL"
            controller_name = "MANUAL"

        pourcentage = round(float(commande_predite * 100), 3)

        print(
            f"[{mode_affichage}] "
            f"Temp: {temp_actuelle:.1f} C | "
            f"Hum: {humidite:.1f}% | "
            f"Consigne: {CONSIGNE} C -> "
            f"Commande: {pourcentage:.3f}%"
        )

        # ==========================================
        # MQTT - Envoi JSON vers superviseur Node-RED
        # ==========================================
        temps_actuel = time.time()

        if (temps_actuel - dernier_envoi_mqtt) >= PERIODE_MQTT:

            payload_mqtt = {
                "temperature": round(float(temp_actuelle), 2),
                "humidity": round(float(humidite), 2),
                "setpoint": round(float(CONSIGNE), 2),
                "command": round(float(pourcentage), 3),
                "mode": MODE.upper(),
                "controller": controller_name,
                "mqtt": "Connected",
                "sensor": sensor_status,
                "influx": "OK",
                "alert": "SURCHAUFFE" if temp_actuelle > CONSIGNE + 5 else "NORMAL",
                "pwm_period": 1.0,
                "pwm_on": round(float(commande_predite), 3),
                "pwm_off": round(float(1.0 - commande_predite), 3),
                "timestamp": time.strftime("%Y-%m-%d %H:%M:%S")
            }

            client_mqtt.publish(MQTT_TOPIC_DATA, json.dumps(payload_mqtt))
            dernier_envoi_mqtt = temps_actuel

        # ==========================================
        # Application de la commande par Slow PWM sur une période de 1 seconde
        # ==========================================
        temps_on = float(commande_predite)
        temps_off = float(1.0 - commande_predite)

        if temps_on > 0:
            GPIO.output(PIN_CHAUFFAGE, GPIO.HIGH)
            time.sleep(temps_on)

        if temps_off > 0:
            GPIO.output(PIN_CHAUFFAGE, GPIO.LOW)
            time.sleep(temps_off)

        temp_t2 = temp_t1
        temp_t1 = temp_actuelle


except KeyboardInterrupt:
    print("\nArret de la regulation.")


finally:
    GPIO.output(PIN_CHAUFFAGE, GPIO.LOW)
    GPIO.cleanup()
    dhtDevice.exit()

    client_mqtt.loop_stop()
    client_mqtt.disconnect()

    print("Systeme eteint proprement.")